# Neural Network Fitting and Benchmarking

## Imports

In [1]:
%load_ext autoreload
%autoreload 2

import os
from m3util.viz.printing import printer
from m3util.ml.optimizers.TrustRegion import TRCG
from m3util.viz.style import set_style
from m3util.ml.rand import set_seeds
from belearn.dataset.dataset import BE_Dataset
from belearn.functions.sho import SHO_nn
from belearn.nn.nn import BatchTrainer
from belearn.nn.inference import BEInference
from datafed_torchflow.datafed import DataFed
from datafed_torchflow.pytorch import TorchViewer
from datetime import datetime

from autophyslearn.postprocessing.complex import ComplexPostProcessor
from autophyslearn.spectroscopic.nn import Multiscale1DFitter, Model


In [2]:
os.environ['CUDA_VISIBLE_DEVICES'] = '1'

# Specify the filename and the path to save the file
filename = "./data_raw.h5"
save_path = "./Data"


optimizer_TR = {"name": "TRCG", "optimizer": TRCG, "radius": 5, "device": "cuda", "ADAM_epochs": 2}
optimizers = ['Adam', optimizer_TR]
noise_list = [0, 1, 2, 3, 4, 5, 6, 7, 8]
batch_size = [500, 1000, 5000, 10000]
epochs = [5]
seed = [41, 43, 44, 45, 46]
early_stopping_time = 60*3
basepath_postfix = 'nn_benchmarks_noise'

# Original filename
csv_name = 'nn_benchmarks_noise.csv'

printing = printer(basepath='./Figures/')

set_style("printing")
set_seeds(seed=42)

# data_path = save_path + "/" + filename
data_path = save_path + "/" + filename

printing set for seaborn
Pytorch seed was set to 42
Numpy seed was set to 42


2025-04-23 12:18:08.045243: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1745425088.060805 3850516 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1745425088.065640 3850516 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-04-23 12:18:08.081456: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


tensorflow seed was set to 42


In [3]:
# instantiate the dataset object
dataset = BE_Dataset(data_path, 
                     #SHO_fit_func_LSQF=SHO_nn, 
                     #datafed = "2024_SHO_Fitting/Training_Benchmarks_NN_SHO_10_21_2024"
                     )

# print the contents of the file
dataset.print_be_tree

/
├ Measurement_000
  ---------------
  ├ Channel_000
    -----------
    ├ Bin_FFT
    ├ Bin_Frequencies
    ├ Bin_Indices
    ├ Bin_Step
    ├ Bin_Wfm_Type
    ├ Excitation_Waveform
    ├ Noise_Floor
    ├ Noisy_Data_1
    ├ Noisy_Data_2
    ├ Noisy_Data_3
    ├ Noisy_Data_4
    ├ Noisy_Data_5
    ├ Noisy_Data_6
    ├ Noisy_Data_7
    ├ Noisy_Data_8
    ├ Position_Indices
    ├ Position_Values
    ├ Raw_Data
    ├ Raw_Data_Reshaped
    ├ Spatially_Averaged_Plot_Group_000
      ---------------------------------
      ├ Bin_Frequencies
      ├ Max_Response
      ├ Mean_Spectrogram
      ├ Min_Response
      ├ Spectroscopic_Parameter
      ├ Step_Averaged_Response
    ├ Spatially_Averaged_Plot_Group_001
      ---------------------------------
      ├ Bin_Frequencies
      ├ Max_Response
      ├ Mean_Spectrogram
      ├ Min_Response
      ├ Spectroscopic_Parameter
      ├ Step_Averaged_Response
    ├ Spectroscopic_Indices
    ├ Spectroscopic_Values
    ├ UDVS
    ├ UDVS_Indices
  ├ Raw_D

### trainer.run_training wants to use test_train_split_, which is in BE_model_utils.py NOT dataset. Therefore I need to instantiate BE_viz even though this notebook doesn't plot anything. This is something not great about the way JGoddy organized stuff

In [4]:
from belearn.viz.viz import Viz


In [5]:
BE_viz = Viz(dataset, printing, verbose=True)
BE_viz.SHO_preprocessing()
BE_viz.SHO_Scaler()

In [7]:
batch_training = True

# CHANGE THIS PATH TO PUT THE DATA IN A NEW DATAFED COLLECTION
datafed_path = "2024_SHO_Fitting/Training_Benchmarks_NN_SHO_04_23_2025"

# Get the current date and time
current_datetime = datetime.now()

# Format the date and time in a 'pretty' format (e.g., YYYY-MM-DD_HH-MM-SS)
formatted_datetime = current_datetime.strftime('%Y-%m-%d_%H-%M-%S')

basepath = f'{formatted_datetime}_{basepath_postfix}'

trainer = BatchTrainer(
    dataset=BE_viz, #should this be dataset or BE_viz?
    optimizers=optimizers,
    noise_list=noise_list,
    batch_size=batch_size,
    epochs=epochs,
    seed=seed,
    basepath=basepath,
    datafed_path=datafed_path,
    script_path=f"{os.getcwd()}/5_nn_fitting_all.ipynb",
    early_stopping_loss=None,
    early_stopping_count=None,
    early_stopping_time=early_stopping_time,
    write_CSV="Batch_Trainging_SpeedTest.csv",
)

if batch_training == True:
    trainer.run_training(BE_viz) 

Pytorch seed was set to 41
Numpy seed was set to 41
tensorflow seed was set to 41

            Dataset = BE_Dataset(file='./Data/./data_raw.h5', noise=0, resampled_bins=None, resampled_data=None, datafed=None, basegroup='/Measurement_000/Channel_000', raw_data_path='Raw_Data_SHO_Fit/Raw_Data-SHO_Fit_000', measurement_data_path='Measurement_Data/Measurement_Data-000', measurement='Measurement_000', SHO_fit_relative_base_path='SHO_Fit_000', SHO_hysteresis_loop_fit_name='Fit-Loop_Fit_000', SHO_hysteresis_loop_guess_name='Guess-Loop_Fit_000')
            Resample = False
            Raw Format = complex
            Fitter = LSQF
            Scaled = False
            Output Shape = pixels
            Measurement State = all
            Resample Resampled = False
            Resample Bins = 165
            LSQF Phase Shift = None
            NN Phase Shift = None
            Noise Level = 0
            Loop Interpolated = False
            
data type <class 'numpy.ndarray'>

            Dat

KeyboardInterrupt: 

In [18]:
os.environ['CUDA_VISIBLE_DEVICES'] = '1'

# Specify the filename and the path to save the file
filename = "./data_raw.h5"
save_path = "./Data"


optimizer_TR = {"name": "TRCG", "optimizer": TRCG, "radius": 5, "device": "cuda", "ADAM_epochs": 2}
optimizers = ['Adam']
noise_list = [0]
batch_size = [10000]
epochs = [1]
seed = [41]
early_stopping_time = 60*3
basepath_postfix = 'nn_benchmarks_noise'

# Original filename
csv_name = 'nn_benchmarks_noise.csv'

printing = printer(basepath='./Figures/')

set_style("printing")
set_seeds(seed=42)

# data_path = save_path + "/" + filename
data_path = save_path + "/" + filename

printing set for seaborn
Pytorch seed was set to 42
Numpy seed was set to 42
tensorflow seed was set to 42


In [19]:
batch_training = True

# CHANGE THIS PATH TO PUT THE DATA IN A NEW DATAFED COLLECTION
datafed_path = "2024_SHO_Fitting/Training_Benchmarks_NN_SHO_04_22_2025"

# Get the current date and time
current_datetime = datetime.now()

# Format the date and time in a 'pretty' format (e.g., YYYY-MM-DD_HH-MM-SS)
formatted_datetime = current_datetime.strftime('%Y-%m-%d_%H-%M-%S')

basepath = f'{formatted_datetime}_{basepath_postfix}'

trainer = BatchTrainer(
    dataset=BE_viz, #should this be dataset or BE_viz?
    optimizers=optimizers,
    noise_list=noise_list,
    batch_size=batch_size,
    epochs=epochs,
    seed=seed,
    basepath=basepath,
    datafed_path=datafed_path,
    script_path=f"{os.getcwd()}/5_nn_fitting_all.ipynb",
    early_stopping_loss=None,
    early_stopping_count=None,
    early_stopping_time=early_stopping_time,
    write_CSV="Batch_Trainging_SpeedTest.csv",
)

if batch_training == True:
    trainer.run_training(BE_viz,save_all=True) 

Pytorch seed was set to 41
Numpy seed was set to 41
tensorflow seed was set to 41

            Dataset = BE_Dataset(file='./Data/./data_raw.h5', noise=0, resampled_bins=None, resampled_data=None, datafed=None, basegroup='/Measurement_000/Channel_000', raw_data_path='Raw_Data_SHO_Fit/Raw_Data-SHO_Fit_000', measurement_data_path='Measurement_Data/Measurement_Data-000', measurement='Measurement_000', SHO_fit_relative_base_path='SHO_Fit_000', SHO_hysteresis_loop_fit_name='Fit-Loop_Fit_000', SHO_hysteresis_loop_guess_name='Guess-Loop_Fit_000')
            Resample = False
            Raw Format = complex
            Fitter = LSQF
            Scaled = False
            Output Shape = pixels
            Measurement State = all
            Resample Resampled = False
            Resample Bins = 165
            LSQF Phase Shift = None
            NN Phase Shift = None
            Noise Level = 0
            Loop Interpolated = False
            
data type <class 'numpy.ndarray'>

            Dat

In [ ]:
# CHANGE THE COLLECTION ID TO THE COLLECTION CORRESPONDING TO THE DATAFED PATH ABOVE
BE_viz.datafed_obj.replace_missing_records(collection_id='c/525610781',
                                            file_path ='/home/jg3837/DataFed_TorchFlow/DataFed_TorchFlow/SHO_example/5_nn_fitting_all.ipynb',
                                            )

# The below was in Josh's notebook. I'm leaving them here in for reference but I didn't look at them

In [ ]:
batch_training = True
datafed_path = "2024_SHO_Fitting/Training_Benchmarks_NN_SHO_9_22_2024"

torch_viewer = TorchViewer(datafed_path)

pd = torch_viewer.getModelCheckpoints(excluded_keys=["script", "Measurement_000"])

pd.head()

In [ ]:
inference_ = True
BE_viz.measurement_state = "all"


if inference_: 
    inference = BEInference(
        pd, BE_viz, df_api=DataFed(datafed_path), root_directory="./Trained Models"
    )
    
    inference.run()